In [ ]:
# ============================================================
# NLP TEXT PREPROCESSING
# Dataset: SMS Spam Collection
# ============================================================

# 1. Install contractions
!pip install contractions -q


# 2. Import libraries
import pandas as pd
import re
import nltk
import contractions
import spacy

from nltk.stem import PorterStemmer
from IPython.display import display


# ============================================================
# 3. DOWNLOAD NLTK RESOURCES
# ============================================================

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


# ============================================================
# 4. LOAD SPACY ENGLISH MODEL
# ============================================================

nlp = spacy.load("en_core_web_sm")


# ============================================================
# 5. INITIALIZE STEMMER
# ============================================================

stemmer = PorterStemmer()


# ============================================================
# 6. READ SMS SPAM COLLECTION DATASET
# ============================================================

df = pd.read_csv(
    "spam.csv",
    encoding="latin-1"
)


# ============================================================
# 7. SELECT USEFUL COLUMNS
# ============================================================

df = df[['v1', 'v2']]
df.columns = ['label', 'message']


# ============================================================
# 8. REMOVE MISSING VALUES
# ============================================================

df.dropna(
    subset=['message'],
    inplace=True
)


# ============================================================
# 9. REMOVE DUPLICATE MESSAGES
# ============================================================

df.drop_duplicates(
    subset=['message'],
    inplace=True
)


# ============================================================
# 10. KEEP ORIGINAL MESSAGE
# ============================================================

df['original_message'] = df['message']


# ============================================================
# TEXT CLEANING
# ============================================================

# 11. Convert text to lowercase

df['message'] = df['message'].str.lower()


# 12. Expand contractions

df['message'] = df['message'].apply(
    contractions.fix
)


# 13. Remove punctuation and numbers

df['message'] = df['message'].apply(
    lambda x: re.sub(
        r'[^a-zA-Z\s]',
        '',
        x
    )
)


# 14. Remove extra whitespace

df['message'] = df['message'].apply(
    lambda x: re.sub(
        r'\s+',
        ' ',
        x
    ).strip()
)


# ============================================================
# TOKENIZATION
# ============================================================

df['tokens'] = df['message'].apply(
    nltk.word_tokenize
)


# ============================================================
# STOP-WORD REMOVAL
# ============================================================

stop_words = set(
    nltk.corpus.stopwords.words('english')
)

df['tokens'] = df['tokens'].apply(
    lambda x: [
        word
        for word in x
        if word not in stop_words
    ]
)


# ============================================================
# LEMMATIZATION
# ============================================================

def lemmatize_words(words):

    text = " ".join(words)

    doc = nlp(text)

    return [
        word.lemma_
        for word in doc
        if not word.is_space
    ]


df['lemmatized_tokens'] = df['tokens'].apply(
    lemmatize_words
)


# ============================================================
# STEMMING
# ============================================================

df['stemmed_tokens'] = df['tokens'].apply(
    lambda x: [
        stemmer.stem(word)
        for word in x
    ]
)


# ============================================================
# CONVERT TOKENS BACK TO TEXT
# ============================================================

df['lemmatized_text'] = df[
    'lemmatized_tokens'
].apply(
    lambda x: " ".join(x)
)


df['stemmed_text'] = df[
    'stemmed_tokens'
].apply(
    lambda x: " ".join(x)
)


# ============================================================
# FINAL TABLE — ONLY 5 ROWS
# ============================================================

print("\nORIGINAL DATA VS PREPROCESSED DATA\n")

final_data = df[
    [
        'label',
        'original_message',
        'message',
        'tokens',
        'lemmatized_text',
        'stemmed_text'
    ]
].head(5).copy()


# Rename columns

final_data.columns = [
    'Label',
    'Original SMS',
    'Cleaned SMS',
    'Tokens',
    'Lemmatized Text',
    'Stemmed Text'
]


# Display table

display(
    final_data.style
    .set_properties(
        **{
            'text-align': 'left',
            'white-space': 'pre-wrap'
        }
    )
    .set_table_styles([
        {
            'selector': 'th',
            'props': [
                ('background-color', '#2E7D32'),
                ('color', 'white'),
                ('font-weight', 'bold'),
                ('text-align', 'center')
            ]
        },
        {
            'selector': 'td',
            'props': [
                ('vertical-align', 'top')
            ]
        }
    ])
)


# ============================================================
# SAVE PREPROCESSED DATASET
# ============================================================

df.to_csv(
    "preprocessed_sms.csv",
    index=False
)

print("\nPreprocessing completed successfully!")
print("Saved as: preprocessed_sms.csv")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



ORIGINAL DATA VS PREPROCESSED DATA



,Label,Original SMS,Cleaned SMS,Tokens,Lemmatized Text,Stemmed Text
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat,"['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']",go jurong point crazy available bugis n great world la e buffet cine get amore wat,go jurong point crazi avail bugi n great world la e buffet cine got amor wat
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif you oni,"['ok', 'lar', 'joking', 'wif', 'oni']",ok lar joking wif oni,ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,free entry in a wkly comp to win fa cup final tkts st may text fa to to receive entry questionstd txt ratetcs apply overs,"['free', 'entry', 'wkly', 'comp', 'win', 'fa', 'cup', 'final', 'tkts', 'st', 'may', 'text', 'fa', 'receive', 'entry', 'questionstd', 'txt', 'ratetcs', 'apply', 'overs']",free entry wkly comp win fa cup final tkts st may text fa receive entry questionstd txt ratetcs apply over,free entri wkli comp win fa cup final tkt st may text fa receiv entri questionstd txt ratetc appli over
3,ham,U dun say so early hor... U c already then say...,you dun say so early hor you c already then say,"['dun', 'say', 'early', 'hor', 'c', 'already', 'say']",dun say early hor c already say,dun say earli hor c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives around here though",nah i do not think he goes to usf he lives around here though,"['nah', 'think', 'goes', 'usf', 'lives', 'around', 'though']",nah think go usf life around though,nah think goe usf live around though



Preprocessing completed successfully!
Saved as: preprocessed_sms.csv


In [ ]:
!pip install -q gensim

import numpy as np
import pandas as pd
from gensim.models import Word2Vec, FastText
import gensim.downloader as api


# ============================================================
# PREPARE SENTENCES
# ============================================================

sentences = df['tokens'].tolist()


# ============================================================
# TRAIN WORD2VEC
# ============================================================

print("Training Word2Vec...")

word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=42
)

print("Word2Vec completed.")


# ============================================================
# TRAIN FASTTEXT
# ============================================================

print("\nTraining FastText...")

fasttext_model = FastText(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=42
)

print("FastText completed.")


# ============================================================
# LOAD GLOVE
# ============================================================

print("\nLoading GloVe...")

glove_model = api.load("glove-wiki-gigaword-100")

print("GloVe completed.")


# ============================================================
# WORD2VEC EMBEDDING FUNCTION
# ============================================================

def word2vec_embedding(tokens):
    vectors = []

    for word in tokens:
        if word in word2vec_model.wv:
            vectors.append(word2vec_model.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)


# ============================================================
# GLOVE EMBEDDING FUNCTION
# ============================================================

def glove_embedding(tokens):
    vectors = []

    for word in tokens:
        if word in glove_model:
            vectors.append(glove_model[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)


# ============================================================
# FASTTEXT EMBEDDING FUNCTION
# ============================================================

def fasttext_embedding(tokens):
    vectors = []

    for word in tokens:
        vectors.append(fasttext_model.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)


# ============================================================
# CREATE EMBEDDINGS
# ============================================================

print("\nCreating embeddings...")

df['Word2Vec_Embedding'] = df['tokens'].apply(
    word2vec_embedding
)

df['GloVe_Embedding'] = df['tokens'].apply(
    glove_embedding
)

df['FastText_Embedding'] = df['tokens'].apply(
    fasttext_embedding
)

print("Embeddings created successfully.")


# ============================================================
# STATIC WORD EMBEDDINGS COMPARISON
# ============================================================

print("\n")
print("=" * 60)
print("STATIC WORD EMBEDDINGS COMPARISON")
print("=" * 60)

comparison = pd.DataFrame({
    'Method': [
        'Word2Vec',
        'GloVe',
        'FastText'
    ],

    'Type': [
        'Static',
        'Static',
        'Static'
    ],

    'Dimension': [
        100,
        100,
        100
    ],

    'Training': [
        'SMS Dataset',
        'Pre-trained',
        'SMS Dataset'
    ],

    'Advantage': [
        'Learns SMS-specific context',
        'Large pre-trained vocabulary',
        'Handles subwords and unknown words'
    ]
})

display(comparison)


# ============================================================
# FIRST SMS EMBEDDING
# ============================================================

print("\n")
print("=" * 60)
print("FIRST SMS EMBEDDING")
print("=" * 60)

print("\nOriginal SMS:")
print(df['original_message'].iloc[0])

print("\nWord2Vec:")
print(df['Word2Vec_Embedding'].iloc[0][:10])

print("\nGloVe:")
print(df['GloVe_Embedding'].iloc[0][:10])

print("\nFastText:")
print(df['FastText_Embedding'].iloc[0][:10])


# ============================================================
# EMBEDDING SHAPES
# ============================================================

print("\n")
print("=" * 60)
print("EMBEDDING SHAPES")
print("=" * 60)

print(
    "Word2Vec:",
    df['Word2Vec_Embedding'].iloc[0].shape
)

print(
    "GloVe:",
    df['GloVe_Embedding'].iloc[0].shape
)

print(
    "FastText:",
    df['FastText_Embedding'].iloc[0].shape
)


# ============================================================
# PREPARE DATA FOR CSV
# ============================================================

embedding_data = df[
    [
        'label',
        'original_message',
        'Word2Vec_Embedding',
        'GloVe_Embedding',
        'FastText_Embedding'
    ]
].copy()


# ============================================================
# CONVERT EMBEDDINGS TO STRINGS
# ============================================================

embedding_data['Word2Vec_Embedding'] = embedding_data[
    'Word2Vec_Embedding'
].apply(
    lambda x: ' '.join(map(str, x))
)

embedding_data['GloVe_Embedding'] = embedding_data[
    'GloVe_Embedding'
].apply(
    lambda x: ' '.join(map(str, x))
)

embedding_data['FastText_Embedding'] = embedding_data[
    'FastText_Embedding'
].apply(
    lambda x: ' '.join(map(str, x))
)


# ============================================================
# SAVE TO CSV
# ============================================================

embedding_data.to_csv(
    'sms_static_embeddings.csv',
    index=False
)

print("\nStatic embeddings saved successfully!")
print("File: sms_static_embeddings.csv")

Training Word2Vec...
Word2Vec completed.

Training FastText...
FastText completed.

Loading GloVe...
[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe completed.

Creating embeddings...
Embeddings created successfully.


STATIC WORD EMBEDDINGS COMPARISON


,Method,Type,Dimension,Training,Advantage
0,Word2Vec,Static,100,SMS Dataset,Learns SMS-specific context
1,GloVe,Static,100,Pre-trained,Large pre-trained vocabulary
2,FastText,Static,100,SMS Dataset,Handles subwords and unknown words




FIRST SMS EMBEDDING

Original SMS:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Word2Vec:
[ 0.09800008 -0.00934729 -0.19504353  0.06569139 -0.06181325  0.11564087
  0.19738682  0.05700374 -0.07148731 -0.00924163]

GloVe:
[-0.05918936  0.07337585  0.25856537 -0.02353659 -0.15043531  0.11440406
  0.04923962  0.24415669  0.02678226 -0.12291642]

FastText:
[ 0.01625435  0.22351998  0.13087061  0.09861332 -0.13344696  0.02596775
  0.39100176 -0.17335823 -0.02770683 -0.0997469 ]


EMBEDDING SHAPES
Word2Vec: (100,)
GloVe: (100,)
FastText: (100,)

Static embeddings saved successfully!
File: sms_static_embeddings.csv
